# Chapitre 1 — Réseaux profonds et calcul différentiable
## Atelier corrigé · M2 IA
Exécutez de haut en bas. Dépendances : NumPy, Matplotlib et PyTorch CPU.
Prédire → tester → expliquer → transférer. Les données sont générées localement.


## 1. Données et protocole
Prédisez la géométrie des classes. Pourquoi normaliser à partir du train uniquement ?


In [ ]:
import numpy as np

# Cellule 1 : données, séparation et normalisation sans fuite.
rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, size=(600, 2))
y = (X[:, 0] * X[:, 1] > 0).astype(np.float64).reshape(-1, 1)
perm = rng.permutation(len(X))
train_idx, val_idx = perm[:450], perm[450:]
mu, sigma = X[train_idx].mean(0), X[train_idx].std(0)
Xtr, Xva = (X[train_idx] - mu) / sigma, (X[val_idx] - mu) / sigma
ytr, yva = y[train_idx], y[val_idx]
print('Formes :', Xtr.shape, ytr.shape, Xva.shape, yva.shape)



## 2. Implémentation du MLP
Annotez chaque forme. Implémentez les fonctions avant de lire la correction.


In [ ]:
# Cellule 2 : paramètres et passage avant.
def init_params(seed=7, hidden=8):
    r = np.random.default_rng(seed)
    return {'W1': r.normal(0, .4, (2, hidden)),
            'b1': np.zeros((1, hidden)),
            'W2': r.normal(0, .4, (hidden, 1)),
            'b2': np.zeros((1, 1))}

def sigmoid(z):
    # Forme stable : aucun exp d'un nombre positif de grande amplitude.
    return np.exp(-np.logaddexp(0., -z))

def forward(X, p):
    H = np.tanh(X @ p['W1'] + p['b1'])
    Z = H @ p['W2'] + p['b2']
    return Z, H

def loss_and_grads(X, y, p):
    Z, H = forward(X, p)
    loss = np.mean(np.logaddexp(0., Z) - y * Z)
    D = (sigmoid(Z) - y) / len(X)
    gW2, gb2 = H.T @ D, D.sum(axis=0, keepdims=True)
    DH = (D @ p['W2'].T) * (1 - H**2)
    g = {'W1': X.T @ DH, 'b1': DH.sum(axis=0, keepdims=True),
         'W2': gW2, 'b2': gb2}
    return float(loss), g

p = init_params()
loss, grads = loss_and_grads(Xtr, ytr, p)
for k in p:
    assert p[k].shape == grads[k].shape
print('Perte initiale :', loss)



## 3. Vérification des gradients
Injectez ensuite une erreur locale et observez quels gradients échouent.


In [ ]:
# Cellule 3 : vérification des gradients en float64.
def gradient_check(X, y, p, eps=1e-5, loss_fn=None):
    loss_fn = loss_fn or loss_and_grads
    _, analytical = loss_fn(X, y, p)
    errors = {}
    for name, param in p.items():
        numerical = np.zeros_like(param)
        for index in np.ndindex(param.shape):
            old = param[index]
            try:
                param[index] = old + eps
                lp, _ = loss_fn(X, y, p)
                param[index] = old - eps
                lm, _ = loss_fn(X, y, p)
                numerical[index] = (lp - lm) / (2 * eps)
            finally:
                param[index] = old
        a = analytical[name]
        errors[name] = np.linalg.norm(a - numerical) / max(
            1e-12, np.linalg.norm(a) + np.linalg.norm(numerical))
    return errors

errors = gradient_check(Xtr[:12], ytr[:12], p)
print('Erreurs relatives :', errors)
assert max(errors.values()) < 1e-6



## 4. Apprentissage et référence linéaire
Interprétez la perte et les deux accuracy. Répétez sur plusieurs graines en prolongement.


In [ ]:
# Cellule 4 : entraînement et référence linéaire.
def accuracy(X, y, p):
    Z, _ = forward(X, p)
    return float(((Z >= 0) == y).mean())

initial = {k: v.copy() for k, v in p.items()}
history = []
eta = .15
for epoch in range(2000):
    loss, g = loss_and_grads(Xtr, ytr, p)
    for k in p:
        p[k] -= eta * g[k]
    if epoch % 100 == 0:
        history.append((epoch, loss, accuracy(Xtr, ytr, p), accuracy(Xva, yva, p)))
print('Dernier relevé (avant mise à jour pour la perte) :', history[-1])
print('Accuracy finale train / validation :', accuracy(Xtr, ytr, p), accuracy(Xva, yva, p))

w, b = np.zeros((2, 1)), np.zeros((1, 1))
for epoch in range(2000):
    z = Xtr @ w + b
    d = (sigmoid(z) - ytr) / len(Xtr)
    w -= eta * (Xtr.T @ d)
    b -= eta * d.sum(axis=0, keepdims=True)
print('Référence linéaire, accuracy validation :', ((Xva @ w + b >= 0) == yva).mean())



## 5. Visualisation
Repérez les zones d’incertitude et comparez avec la frontière réelle.


In [ ]:
# Cellule 5 : graphique (optionnel, nécessite matplotlib).
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot([h[0] for h in history], [h[1] for h in history])
axes[0].set(xlabel='Époque', ylabel='Perte train', title='Objectif entraînement')
grid_x, grid_y = np.meshgrid(np.linspace(-2, 2, 150), np.linspace(-2, 2, 150))
raw_grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])
zg, _ = forward((raw_grid - mu) / sigma, p)
axes[1].contourf(grid_x, grid_y, sigmoid(zg).reshape(grid_x.shape), levels=20, cmap='RdBu', alpha=.6)
axes[1].scatter(X[val_idx, 0], X[val_idx, 1], c=yva.ravel(), cmap='RdBu', s=10, edgecolor='k', linewidth=.2)
axes[1].set(title='Probabilité et observations de validation', xlabel='x1', ylabel='x2')
plt.tight_layout()
plt.show()



## 6. Comparaison PyTorch
Prédisez les causes possibles d’un écart : réduction, dtype, initialisation, transposée.


In [ ]:
# Cellule 6 : comparaison PyTorch, à paramètres et précision identiques.
import torch
import torch.nn.functional as F
pt = {k: torch.tensor(v.copy(), dtype=torch.float64, requires_grad=True)
      for k, v in initial.items()}
xt = torch.tensor(Xtr[:12], dtype=torch.float64)
yt = torch.tensor(ytr[:12], dtype=torch.float64)
ht = torch.tanh(xt @ pt['W1'] + pt['b1'])
zt = ht @ pt['W2'] + pt['b2']
lt = F.binary_cross_entropy_with_logits(zt, yt, reduction='mean')
lt.backward()
ln, gn = loss_and_grads(Xtr[:12], ytr[:12], initial)
np.testing.assert_allclose(lt.item(), ln, rtol=1e-10, atol=1e-12)
for k in pt:
    np.testing.assert_allclose(pt[k].grad.numpy(), gn[k], rtol=1e-8, atol=1e-10)
print('NumPy / PyTorch : pertes et tous les gradients concordent.')



## 7. Accumulation et diagnostic
Prédisez chaque valeur avant exécution. Recréez les feuilles avant le défi detach.


In [ ]:
# Cellule 7 : accumulation et rupture du graphe.
a = torch.tensor(2., dtype=torch.float64, requires_grad=True)
(a**2).backward()
first = a.grad.item()
(a**2).backward()  # Nouveau graphe ; même accumulateur .grad.
second = a.grad.item()
print('Gradients accumulés :', first, second)
assert first == 4. and second == 8.
a.grad = None
(a**2).backward()
assert a.grad.item() == 4.
print('Après remise à zéro :', a.grad.item())
# Diagnostic : si ht est remplacé par ht.detach(), quels paramètres perdent leur gradient ?


## 8. Prolongement A · Un moteur d’autodiff maison
Pourquoi l’ordre topologique inverse ? Que se passe-t-il si l’on appelle `backward` deux fois ?


In [ ]:
# Cellule 8 : un moteur d'autodiff scalaire en mode inverse.
import math

class Var:
    """Nombre réel qui mémorise comment il a été calculé."""
    def __init__(self, value, parents=()):
        self.value = float(value)
        self.grad = 0.0
        self.parents = parents  # tuples (nœud parent, dérivée locale)

    def __add__(self, other):
        other = other if isinstance(other, Var) else Var(other)
        return Var(self.value + other.value, ((self, 1.0), (other, 1.0)))
    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Var) else Var(other)
        return Var(self.value * other.value, ((self, other.value), (other, self.value)))
    __rmul__ = __mul__

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return (-self) + other

    def tanh(self):
        t = math.tanh(self.value)
        return Var(t, ((self, 1.0 - t * t),))

    def softplus(self):
        # log(1 + e^z) calculé de façon stable ; sa dérivée est σ(z).
        z = self.value
        s = 1 / (1 + math.exp(-z)) if z >= 0 else math.exp(z) / (1 + math.exp(z))
        return Var(max(z, 0.) + math.log1p(math.exp(-abs(z))), ((self, s),))

    def backward(self):
        order, seen = [], set()
        def visit(node):  # tri topologique : parents avant enfants
            if node not in seen:
                seen.add(node)
                for parent, _ in node.parents:
                    visit(parent)
                order.append(node)
        visit(self)
        self.grad = 1.0
        for node in reversed(order):
            for parent, local in node.parents:
                parent.grad += local * node.grad  # les branches se somment

# Vérification 1 : exercice 5 du TD, L = 2ab − a − b en (2, 4).
a, b = Var(2.), Var(4.)
L = 2 * a * b - a - b
L.backward()
print('∂L/∂a, ∂L/∂b :', a.grad, b.grad)
assert (a.grad, b.grad) == (7., 3.)

# Vérification 2 : le MLP complet, comparé à loss_and_grads.
def mlp_loss_var(X, y, P):
    total = 0.
    for xi, yi in zip(X.tolist(), y.ravel().tolist()):
        h = [(sum(xi[j] * P['W1'][j][k] for j in range(2)) + P['b1'][0][k]).tanh()
             for k in range(len(P['b1'][0]))]
        z = sum(h[k] * P['W2'][k][0] for k in range(len(h))) + P['b2'][0][0]
        total = total + z.softplus() - yi * z
    return total * (1 / len(X))

P = {k: [[Var(v) for v in row] for row in arr.tolist()] for k, arr in initial.items()}
Lv = mlp_loss_var(Xtr[:12], ytr[:12], P)
Lv.backward()
ln, gn = loss_and_grads(Xtr[:12], ytr[:12], initial)
assert abs(Lv.value - ln) < 1e-12
for k in P:
    np.testing.assert_allclose([[v.grad for v in row] for row in P[k]], gn[k], rtol=1e-10, atol=1e-12)
print('Moteur maison / NumPy : perte et 33 gradients concordent.')


## 9. Prolongement B · Softmax à trois classes et mini-lots
Vérifiez d’abord les gradients, puis comparez à la classe majoritaire.


In [ ]:
# Cellule 9 : softmax à trois classes et descente par mini-lots.
def make_rings(n, seed):
    r = np.random.default_rng(seed)
    X = r.uniform(-2, 2, size=(n, 2))
    return X, np.digitize(np.linalg.norm(X, axis=1), [0.9, 1.6])  # classes 0, 1, 2

X3, y3 = make_rings(900, seed=0)
mu3, sd3 = X3[:675].mean(0), X3[:675].std(0)
X3tr, X3va = (X3[:675] - mu3) / sd3, (X3[675:] - mu3) / sd3
y3tr, y3va = y3[:675], y3[675:]

def softmax_loss_and_grads(X, y, p):
    H = np.tanh(X @ p['W1'] + p['b1'])
    Z = H @ p['W2'] + p['b2']                       # logits B×K
    lse = np.logaddexp.reduce(Z, axis=1, keepdims=True)
    rows = np.arange(len(X))
    loss = np.mean(lse[:, 0] - Z[rows, y])
    D = np.exp(Z - lse)                             # probabilités p
    D[rows, y] -= 1                                 # p − t
    D /= len(X)
    DH = (D @ p['W2'].T) * (1 - H**2)
    return float(loss), {'W1': X.T @ DH, 'b1': DH.sum(0, keepdims=True),
                         'W2': H.T @ D, 'b2': D.sum(0, keepdims=True)}

def init_params_k(seed=3, hidden=16, k=3):
    r = np.random.default_rng(seed)
    return {'W1': r.normal(0, .5, (2, hidden)), 'b1': np.zeros((1, hidden)),
            'W2': r.normal(0, .5, (hidden, k)), 'b2': np.zeros((1, k))}

def accuracy_k(X, y, p):
    return float((np.argmax(np.tanh(X @ p['W1'] + p['b1']) @ p['W2'] + p['b2'], 1) == y).mean())

p3 = init_params_k()
errors3 = gradient_check(X3tr[:10], y3tr[:10], p3, loss_fn=softmax_loss_and_grads)
print('Erreurs relatives (softmax) :', errors3)
assert max(errors3.values()) < 1e-6

batch, eta3, order = 32, .5, np.random.default_rng(1)
for epoch in range(200):
    idx = order.permutation(len(X3tr))              # nouvel ordre à chaque époque
    for start in range(0, len(idx), batch):
        sel = idx[start:start + batch]
        _, g = softmax_loss_and_grads(X3tr[sel], y3tr[sel], p3)
        for k in p3:
            p3[k] -= eta3 * g[k]
print('Effectifs des classes (train) :', np.bincount(y3tr))
print('Accuracy train / validation :', accuracy_k(X3tr, y3tr, p3), accuracy_k(X3va, y3va, p3))
